In [ ]:
# unit_test/test_distribution.py
import pytest
import torch
import torch.nn as nn

# ABSOLUTE IMPORT (Works because PYTHONPATH=/app)
from core.distribution import LigandEnvironment

def test_ligand_shapes():
    # 1. Setup
    n_units = 26
    n_families = 5
    batch_size = 10
    
    # 2. Initialize Module
    env = LigandEnvironment(n_units, n_families)
    
    # 3. Run Forward Pass
    energies, concs, fam_ids = env.sample_batch(batch_size)
    
    # 4. Assertions
    assert energies.shape == (batch_size, n_units, 2)
    assert concs.shape == (batch_size,)
    assert torch.all(concs > 0) # Concentration must be positive

In [4]:
import sys
sys.path.append('/app')
# unit_test/test_single_receptor.py
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import os
from itertools import cycle
from src import (LigandEnvironment,
                SymmetricLigandEnvironment,
                BinaryReceptor,
                BaseReceptor,
                generate_receptor_indices, 
                plot_family_summary,
                LogNormalConcentration,
                plot_latent_radar_chart,
                evaluate_model,
                plot_summary,
                plot_latent_umap)
from objectives import DiscreteProxyLoss,TolerantDiscreteProxyLoss,DiscreteExactLoss

In [2]:
n_units=5
k_sub = 5
n_families = 1
N_train = 2**17
N_test = 2**14
CONF = {
        "n_units": n_units,
        "n_families": n_families,
        "latent_dim": 10,
        "k_sub": k_sub,
        "batch_size": N_train,
        "epochs": 5000,
        "lr": 0.05,
        "cov_weight":10.,
        "n_bins":2,
        "bin_temp":0.05,
        "receptor_indices":torch.tensor([[i for _ in range(k_sub)] for i in range(n_units)],dtype=torch.long), # size must be smaller or equal to n_units
        "init_means":[np.random.randint(1,8) for _ in range(n_families)], # size must be n_families
        "shape_sigma": 1.,
        "tolerant":False, # set whether we want tolerance for heteromers on the covariance loss
        "optimizer":"Adam",
        "momentum":0.9,
        "exact_loss":True,
        "temperature":0.1
    }

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
conc_strategy = LogNormalConcentration(n_families=CONF['n_families'], 
                                        init_mean=CONF['init_means'])

env = LigandEnvironment(CONF['n_units'],
                    CONF['n_families'], 
                    conc_model=conc_strategy,
                    latent_dim=CONF['latent_dim'],
                    shape_sigma=CONF.get('shape_sigma', 0.5)).to(device)

In [6]:
env.unit_latent

Parameter containing:
tensor([[ 0.4354,  1.0900,  1.6587, -0.9898, -0.6985, -0.7933, -0.3098,  0.4473,
          0.9403, -1.1695],
        [ 0.4186, -0.5128, -1.0808,  0.5603,  1.0461,  0.3905, -0.4166,  0.3655,
          1.1026, -1.1818],
        [-1.1205, -3.1921, -2.4994,  0.9699,  0.5264, -1.4600,  0.2828, -0.2686,
          1.5252,  0.3098],
        [ 1.3114,  1.5154,  0.9095, -1.3859, -0.8829,  1.7554, -0.8600,  1.8743,
          0.9905,  1.7784],
        [ 0.0832,  0.6509,  0.9297,  0.2363,  1.1451,  0.3964, -0.2914,  0.1039,
         -1.5648, -0.1632]], device='cuda:0', requires_grad=True)

In [7]:
physics = BinaryReceptor(CONF["n_units"], CONF["k_sub"],temperature=CONF["temperature"]).to(device)

In [8]:
energies, concs, _ = env.sample_batch(CONF['batch_size'])
# B. Physics
# activity: (B, 1)
activity = physics(energies, concs, CONF["receptor_indices"])

In [9]:
receptor_indices = CONF['receptor_indices']
flat_indices = receptor_indices.view(-1)
gathered_flat = energies[:, flat_indices]

In [13]:
gathered_flat

tensor([[19.8020, 19.8020, 19.8020,  ..., 29.0132, 29.0132, 29.0132],
        [31.6481, 31.6481, 31.6481,  ..., 34.5143, 34.5143, 34.5143],
        [25.5104, 25.5104, 25.5104,  ..., 20.2060, 20.2060, 20.2060],
        ...,
        [30.3715, 30.3715, 30.3715,  ..., 14.1241, 14.1241, 14.1241],
        [21.7298, 21.7298, 21.7298,  ..., 21.4768, 21.4768, 21.4768],
        [30.9000, 30.9000, 30.9000,  ..., 20.3198, 20.3198, 20.3198]],
       device='cuda:0', grad_fn=<IndexBackward0>)